### Testing Big Query

This is a work in progress. Intended to show how to call functions

In [1]:
import os
from dotenv import load_dotenv
import json
import pandas as pd
import numpy as np
from google.cloud import bigquery
import pydata_google_auth
import requests
import time
from datetime import timedelta
import urllib3

from scripts.queries import *
from scripts.constants import SITE_MAPPINGS, SITE_BURN_DATATYPES, SITE_GENERATION_DATATYPES, AVAILABILITY_FILES_XLSX, AVAILABILITY_FILES_CSV
from scripts.data_pull_functions import run_date_parameterized_query, run_generation_query,  pull_unit_availability, pull_yes_forecast_historical, pull_yes_actual_historical
from scripts.data_clean_functions import clean_generation_unit_data, clean_yes_actual, clean_generation_unit_data, clean_yes_forecast

load_dotenv()

True

In [2]:
# this will open a browser to confirm access when running for the first time
credentials = pydata_google_auth.get_user_credentials(
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
    auth_local_webserver=True,
)

In [3]:
client = bigquery.Client(project="bepc-prj-energy-prod", credentials=credentials )

## Gas Burn Data

In [ ]:
#gas_daily_df = run_date_parameterized_query(client, GAS_BURN_QUERY, '2024-07-23', '2026-07-22', SITE_BURN_DATATYPES)
gas_daily_df = run_date_parameterized_query(client, GAS_BURN_QUERY, '2024-01-01', '2026-07-31', SITE_BURN_DATATYPES)

In [ ]:
gas_daily_df['site'] = gas_daily_df['marketarea'].replace(SITE_MAPPINGS)

gas_daily_df = gas_daily_df[['gas_day', 'site', 'energy']]
gas_daily_df.head()

In [ ]:
# Validity check against 'checkpoint sql 1' file
gas_daily_df.loc[:, ['site', 'energy']].groupby(by=['site']).agg('sum') # matches

## Generation Data

Goal is to write a single function that
1) runs pgs and non-pgs queries using the run_energy_query
2) concatenate the queries
3) make any adjustments to both table at the same time

In [ ]:
## This has been converted to a function in the data_pull_functions.py file
def run_generation_query(client: bigquery.Client, query_strings: list, start: str, end: str, col_dtypes: dict = None):
    """
    Function designed to pull generation data based on multiple SQL queries. SQL queries must contain the same column headers

    Parameters:
        client: Session bigquery.client object
        query_strings: SQL queries
        start: Earliest date of data to pull from database
        end: Latest date of data to pull from database
        col_type: dictionary containing column name to datatype mappings

    """

    dfs = {}
    for idx, query in enumerate(query_strings):
    
        df_name = f'df{idx}'
        dfs[df_name] = run_date_parameterized_query(client, query, start, end)
    
    dfs_appended = pd.concat(dfs, ignore_index=True)

    if col_dtypes: 
        dfs_appended = dfs_appended.astype(col_dtypes)

    return dfs_appended



In [ ]:
df_load_generation = run_generation_query(client, [PGS_GENERATION_QUERY, NON_PGS_GENERATION_QUERY], '2025-01-01', '2025-12-31', SITE_GENERATION_DATATYPES)

#test = test.astype(SITE_GENERATION_DATATYPES)
print(df_load_generation.shape)
#test.head(-5)
#test.to_csv(r'./output-data/non_pgs_only check.csv', index=False)

In [ ]:
df_load_generation[(df_load_generation['loadshape']=='WAUE.BEPM.PGS1 - Net Generation-5m')]

In [ ]:
# This has been converted to a function in the data_pull_functions.py file
def clean_generation_unit_data(df: pd.DataFrame):
    """
    Function that takes the unit generation data as input, creates a site variable, calculates gas day, and aggregates by site

    Parameters:
        df: dataframe containing hourly unit level generation data
    """
    unit_df = df.copy()

    #adding a site column based on the loadshape name
    conditions = [
        unit_df["loadshape"].str.upper().str.contains("DCS"),
        unit_df["loadshape"].str.upper().str.contains("LCS"),
        unit_df["loadshape"].str.upper().str.contains("PGS"),
        unit_df["loadshape"].str.upper().str.contains("GGS"),
        unit_df["loadshape"].str.upper().str.contains("CGS")
    ]

    choices = ["DCS", "LCS", "PGS", "GGS", "CGS"]

    unit_df['site'] = np.select(conditions, choices, default="N/A")

    #Unpivoting columns using melt() function
    hour_cols = [column for column in unit_df.columns if column.startswith("he")]
    hourly_unit_generation_df = unit_df.melt(id_vars=["begtime", "site", "loadshape"], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")
    #print(hourly_unit_generation_df['hourly_mw'].sum()) # may take out

    #Create hour (numeric column) for Datetime creation
    hourly_unit_generation_df["hour_num"] = hourly_unit_generation_df["hour"].str[-2:].astype(int)

    #create a datetime column and related gas_day column
    hourly_unit_generation_df["datetime"] = (pd.to_datetime(hourly_unit_generation_df["begtime"]) + pd.to_timedelta(hourly_unit_generation_df["hour_num"] - 0, unit="h"))
    hourly_unit_generation_df['gas_day'] = (pd.to_datetime(hourly_unit_generation_df["datetime"]) - pd.Timedelta(hours=10)).dt.date
    hourly_unit_generation_df['gas_day'] = pd.to_datetime(hourly_unit_generation_df['gas_day'])
    #print(hourly_unit_generation_df['hourly_mw'].sum()) # may take out


    #Reordering columns for visual 
    hourly_unit_generation_df = (
        hourly_unit_generation_df[["datetime","gas_day", "hour", "site", "loadshape", "hourly_mw"]]
        .sort_values(by = ['site', 'datetime'], ascending=[False, True])
    )
    print(hourly_unit_generation_df['hourly_mw'].sum()) # may take out


    # Aggregating by site
    hourly_site_generation_df = (
        hourly_unit_generation_df.groupby(["datetime", "gas_day", "hour", "site"], as_index=False)["hourly_mw"]
        .sum()
        .rename(columns={"hourly_mw": "hourly_site_gen_mw"})
        .sort_values(by = ['site', 'datetime'], ascending=[False, True])
    )
    #print(hourly_site_generation_df['hourly_site_gen_mw'].sum()) # may take out


    daily_site_generation_by_gas_day_df = (
        hourly_unit_generation_df.groupby(["gas_day", "site"], as_index=False)["hourly_mw"]
        .sum()
        .rename(columns={"hourly_mw": "hourly_site_gen_mw"})
        .sort_values(by = ['site', 'gas_day'], ascending=[False, True])
    )
    #print(daily_site_generation_by_gas_day_df['hourly_site_gen_mw'].sum()) # may take out


    daily_site_generation_df = (
        hourly_unit_generation_df.groupby(["datetime", "site"], as_index=False)["hourly_mw"]
        .sum()
        .rename(columns={"hourly_mw": "hourly_site_gen_mw"})
        .sort_values(by = ['site', 'datetime'], ascending=[False, True])
    )
    #print(daily_site_generation_df['hourly_site_gen_mw'].sum()) # may take out



    return {
        'hourly_unit_generation_df': hourly_unit_generation_df, 
        'hourly_site_generation_df': hourly_site_generation_df, 
        'daily_site_generation_by_gas_day_df': daily_site_generation_by_gas_day_df, 
        'daily_site_generation_df': daily_site_generation_df
    }


In [ ]:
df_load_generation_cleaned = clean_generation_unit_data(df_load_generation)
df_load_generation_cleaned['hourly_site_generation_df']


## Non-PGS Data Generation Data

In [ ]:
## updating column types from the get go
hour_end_columns = [col for col in pgs_generation_df.columns if col.startswith('he')]

pgs_datatypes = {col: 'float64' for col in hour_end_columns}
pgs_datatypes.update({
    'begtime': 'datetime64[ns]', 
    'loadshape': 'str'
})


In [ ]:
pgs_datatypes

In [ ]:

pgs_generation_df = non_pgs_generation_df.astype(pgs_datatypes)

In [ ]:
non_pgs_generation_df['he1'].sum()

## PGS Generation Data

In [ ]:
query_job = client.query(PGS_GENERATION_QUERY)
pgs_generation_df = query_job.to_dataframe()
pgs_generation_df.head()

In [ ]:
hour_end_columns = [col for col in pgs_generation_df.columns if col.startswith('he')]

pgs_datatypes = {col: 'float64' for col in hour_end_columns}
pgs_datatypes.update({
    'begtime': 'datetime64[ns]', 
    'loadshape': 'str'
})

pgs_generation_df = pgs_generation_df.astype(pgs_datatypes)

In [ ]:
pgs_generation_df.info()

In [ ]:
pgs_generation_df['he1'].sum()

## Unit Availability Data Pull

In [6]:
AVAILABILITY_FILES_CSV

['G:\\Trading\\Market Operations\\Unit availability\\2025\\dpm_BEPC_GROUPING_2025030100_2025033123 - March.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2025\\transposed_dpm_BEPC_GROUPING_2025020100_2025022823 feb.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2025\\transposed_dpm_BEPC_GROUPING_2025010100_2025013123.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2024\\dpm_BEPC_GROUPING_2024120100_2024123123.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2024\\dpm_BEPC_GROUPING_2024110100_2024113023.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2024\\dpm_BEPC_GROUPING_2024100100_2024103123.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2024\\dpm_BEPC_GROUPING_2024090100_2024093023.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2024\\dpm_BEPC_GROUPING_2024080100_2024083123.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2024\\dpm_BEPC_GROUPING_2024070100_2024073123.csv',
 'G:\\Trading\\Market

In [45]:
# This function was been moved to the data_pull_functions.py module
def pull_unit_availability(excel_files, csv_files) -> dict:

    excel_dfs = []
    for file in excel_files:
        df = pd.read_excel(file, sheet_name="Gas HEL Transposed")
        df["source_file"] = file
        excel_dfs.append(df)

    excel_combined = pd.concat(excel_dfs, ignore_index=True)

    csv_dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df["source_file"] = file
        csv_dfs.append(df)

    csv_combined = pd.concat(csv_dfs, ignore_index=True)

    #Concat the csvs and the excel files
    combined_df = pd.concat([excel_combined, csv_combined], ignore_index=True)

    #Making all column names lowercase and remove spaces
    combined_df.columns = (combined_df.columns.str.strip().str.lower())

    #Making datetime column in datetime format
    combined_df["datetime"] = pd.to_datetime( combined_df["datetime"], errors="coerce")

    #Sorting chronologically
    combined_df = (combined_df.sort_values("datetime").reset_index(drop=True))


    #Data cleaning and formatting
    combined_df.columns = combined_df.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

    combined_df = combined_df.dropna(subset = ['datetime'], axis=0) # drops rows that don't have dates due to daylight savings shifts
    
    value_cols = [col for col in combined_df.columns if isinstance(col, str) and "high effective limit" in col]

    unit_availability_df = combined_df.melt(id_vars=["datetime"], value_vars=value_cols, var_name="unit", value_name="availability_mw")

    unit_availability_df["site"] = unit_availability_df["unit"].str.extract(r"^(cgs|dcs|ggs|lcs|pgs)", expand=False).str.strip().str.upper()
    unit_availability_df = unit_availability_df.sort_values(by=["site", "unit", "datetime"])[["datetime","site", "unit", "availability_mw"]]

    site_availability_df = unit_availability_df.groupby(["datetime", 'site']).agg({"availability_mw": "sum"}).reset_index()
    site_availability_df = unit_availability_df.sort_values(by=["site", "datetime"])[["datetime","site", "availability_mw"]]

    return {
        "unit_availability_df": unit_availability_df,
        "site_availability_df": site_availability_df
        }

In [50]:
availability_dict = pull_unit_availability(AVAILABILITY_FILES_XLSX, AVAILABILITY_FILES_CSV)

unit_availability_df = availability_dict['unit_availability_df']
site_availability_df = availability_dict['site_availability_df']

In [52]:
site_availability_df["availability_mw"].sum()

np.float64(18991616.34)

In [53]:
site_availability_df.to_csv('./output-data/availability check.csv', index=False)

In [54]:
site_availability_df.head()

,datetime,site,availability_mw
0,2024-05-01 00:00:00,CGS,45.0
1,2024-05-01 01:00:00,CGS,45.0
2,2024-05-01 02:00:00,CGS,45.0
3,2024-05-01 03:00:00,CGS,45.0
4,2024-05-01 04:00:00,CGS,45.0


## Yes Energy Data Pulls
There are two separate data pulls
1. historic/actuals
2. forecasted values

In [5]:
yes_username = os.getenv('YES_USERNAME')
yes_password = os.getenv('YES_PASSWORD')

print(yes_username)
print(yes_password)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

akramer@bepc.com
Basin123


In [ ]:
# This function has been moved to data_pull_functions.py
def pull_yes_forecast_historical(user, password, start_date, end_date):
    """
    Function pulls forecast data from YES Energy via API. Note the hours here are hour ending which is different than Allegro I believe

    Parameters:
        user:
        password:
        start_date:
        end_date:

    Returns:
        Dataframe 
    """
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    df_forecast = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling forecast: {month_start.date()} ---> {month_end.date()}")

        url = ( "https://services.yesenergy.com/PS/rest/timeseries/multiple.json?agglevel=hour&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            "LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_CURRENT:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDFCST_HOURLY:10004185377,"
            "WINDFCST_HOURLY:10004185378,"
            "WINDFCST_HOURLY:10004185379,"
            "WINDFCST_HOURLY:10004185380,"
            "WINDFCST_HOURLY:10004185381,"
            "WSI_FC15_FEEL:10000355230,"
            "WSI_FC15_FEEL:10000355704,"
            "WSI_FC15_FEEL:10000356081,"
            "WSI_FC15_WIND:10000355230,"
            "WSI_FC15_WIND:10000355704" )


        response = requests.get(url, auth=(user, password), verify=False, timeout=120)
        response.raise_for_status()
        
        # processing after successful data pull
        df_chunk = pd.DataFrame(response.json())
       
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        df_forecast.append(df_chunk)

        time.sleep(6)  

        start = month_end + timedelta(days=1)

    return pd.concat(df_forecast, ignore_index=True)

In [ ]:
# This function has been moved to data_pull_functions.py
def clean_yes_forecast(df):

    df = df.copy()

    #find datetime column
    datetime_col = [c for c in df.columns if "DATETIME" in c.upper()][0]

    df["datetime"] = pd.to_datetime(df[datetime_col], format="%m/%d/%Y %H:%M:%S", errors="coerce")
    df["datetime"] = df["datetime"] - pd.Timedelta(hours=1) #YES Energy uses 1:00 in the date time to represent HE1.

    #load 
    df["load_forecast"] = pd.to_numeric( df["SPPISO-East (LOAD_FORECAST)"], errors="coerce")

    #net load 
    df["net_load_forecast"] = pd.to_numeric(df["SPPISO-East (NET_LOAD_FORECAST_CURRENT)"], errors="coerce")

    #wind 
    wind_cols = [c for c in df.columns if "WINDFCST_HOURLY" in c]
    df["wind_forecast"] = df[wind_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    #outages 
    df["offline_ng_forecast"] = pd.to_numeric(df["SPPISO-East (NG_CAPACITY_OFFLINE)"], errors="coerce")

    df["offline_coal_forecast"] = pd.to_numeric(df["SPPISO-East (COAL_CAPACITY_OFFLINE)"], errors="coerce")

    #temperature avg of the zones (ask about this)
    temp_cols = [c for c in df.columns if "WSI_FC15_FEEL" in c]
    df["temperature_forecast"] = df[temp_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    #wind speed avg of the reserve zones (also ask)
    wind_speed_cols = [c for c in df.columns if "WSI_FC15_WIND" in c]
    df["wind_speed_forecast"] = df[wind_speed_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    #outage
    df["total_offline_forecast"] = df["offline_ng_forecast"] + df["offline_coal_forecast"]

    out = df[["datetime", "load_forecast", "net_load_forecast", "wind_forecast", "temperature_forecast", "wind_speed_forecast", "total_offline_forecast", "offline_ng_forecast", "offline_coal_forecast"]]#.dropna(subset=["datetime"])

    return out.sort_values("datetime").reset_index(drop=True)


In [ ]:
start = "2026-07-01"
end = "2026-12-31"

df_forecast = pull_yes_forecast_historical(yes_username, yes_password, start_date=start, end_date = end)

Pulling forecast: 2026-07-01 ---> 2026-07-31
Pulling forecast: 2026-08-01 ---> 2026-08-31
Pulling forecast: 2026-09-01 ---> 2026-09-30
Pulling forecast: 2026-10-01 ---> 2026-10-31
Pulling forecast: 2026-11-01 ---> 2026-11-30
Pulling forecast: 2026-12-01 ---> 2026-12-31


In [7]:
df_forecast.head()

,DATETIME,SPPISO-East (LOAD_FORECAST),SPPISO-East (NET_LOAD_FORECAST_CURRENT),SPPISO-East (NG_CAPACITY_OFFLINE),SPPISO-East (COAL_CAPACITY_OFFLINE),RESERVE ZONE 1 (WINDFCST_HOURLY),RESERVE ZONE 2 (WINDFCST_HOURLY),RESERVE ZONE 3 (WINDFCST_HOURLY),RESERVE ZONE 4 (WINDFCST_HOURLY),RESERVE ZONE 5 (WINDFCST_HOURLY),ND - Bismarck/Municipal (WSI_FC15_FEEL),ND - Fargo/Hector Field (WSI_FC15_FEEL),ND - Williston/Sloulin (WSI_FC15_FEEL),ND - Bismarck/Municipal (WSI_FC15_WIND),ND - Fargo/Hector Field (WSI_FC15_WIND),HOURENDING,MARKETDAY,PEAKTYPE,MONTH,YEAR
0,07/01/2026 01:00:00,38465,12036.39,6500.13,3968,1847.95,5730.45,1970.34,15443.21,1436.66,64.58,70.34,60.44,5,8.1,1,07/01/2026,None,JULY,2026
1,07/01/2026 02:00:00,37009,11144.8,6500.13,3968,1809,5897.96,1837.17,15025.71,1294.36,62.78,68.54,59.54,5,8.1,2,07/01/2026,None,JULY,2026
2,07/01/2026 03:00:00,35846,9385.99,6500.13,3968,1967.57,5894.54,1813.2,15325.41,1459.29,61.16,66.56,58.28,3.7,7.4,3,07/01/2026,None,JULY,2026
3,07/01/2026 04:00:00,35214,9014.47,6500.13,3968,1850.7,6053.33,1804.95,15192.34,1298.21,59.72,64.76,57.2,3.1,6.8,4,07/01/2026,None,JULY,2026
4,07/01/2026 05:00:00,35159,9396.6,6500.13,3968,1519.39,5984.19,1936.6,14897.45,1424.77,59.92,63.88,56.84,5.6,6.8,5,07/01/2026,None,JULY,2026


In [8]:
df_forecast_cleaned = clean_yes_forecast(df_forecast)

In [9]:
df_forecast_cleaned.head()

,datetime,load_forecast,net_load_forecast,wind_forecast,temperature_forecast,wind_speed_forecast,total_offline_forecast,offline_ng_forecast,offline_coal_forecast
0,2026-07-01 00:00:00,38465.0,12036.39,26428.61,65.120000,6.55,10468.13,6500.13,3968.0
1,2026-07-01 01:00:00,37009.0,11144.80,25864.20,63.620000,6.55,10468.13,6500.13,3968.0
2,2026-07-01 02:00:00,35846.0,9385.99,26460.01,62.000000,5.55,10468.13,6500.13,3968.0
3,2026-07-01 03:00:00,35214.0,9014.47,26199.53,60.560000,4.95,10468.13,6500.13,3968.0
4,2026-07-01 04:00:00,35159.0,9396.60,25762.40,60.213333,6.20,10468.13,6500.13,3968.0


In [ ]:
# function has been moved to data_pull_functions.py module
def pull_yes_actual_historical(user, password, start_date, end_date) -> pd.DataFrame:
    """
    Function to pull actual data from the YES Energy API.

    Parameters:
        user: YES Energy API Username
        password: YES Energy API Password
        start_date: First date of data to pull
        end_date: Last date of data to pull

    Returns:
        dataframe with data
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    df_actual = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling actuals: {month_start.date()} ---> {month_end.date()}")

        url = ("https://services.yesenergy.com/PS/rest/timeseries/multiple.json?agglevel=hour&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            #day ahead close so just in actual
            "BIDCLOSE_LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_BID_CLOSE:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDGEN_HOURLY:10004185377,"
            "WINDGEN_HOURLY:10004185378,"
            "WINDGEN_HOURLY:10004185379,"
            "WINDGEN_HOURLY:10004185380,"
            "WINDGEN_HOURLY:10004185381,"
            "WSI_TRADER_FEELS_TEMP:10000355230,"
            "WSI_TRADER_FEELS_TEMP:10000355704,"
            "WSI_TRADER_FEELS_TEMP:10000356081,"
            "WSI_TRADER_WIND:10000355230,"
            "WSI_TRADER_WIND:10000355704")


        response = requests.get(url, auth=(user, password), verify=False, timeout=120)
        #response.raise_for_status()


        df_chunk = pd.DataFrame(response.json())
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        df_actual.append(df_chunk)
        
        time.sleep(6)  

        start = month_end + timedelta(days=1)

    return pd.concat(df_actual, ignore_index=True)



In [ ]:
# function has been moved to data_clean_functions.py module
def clean_yes_actual(df):
  
    df = df.copy()

    datetime_col = [c for c in df.columns if "DATETIME" in c.upper()][0]

    df["datetime"] = pd.to_datetime( df[datetime_col], errors="coerce")
    df["datetime"] = df["datetime"] - pd.Timedelta(hours=1) #YES Energy uses 1:00 in the date time to represent HE1.


    # load actual
    df["load_actual"] = pd.to_numeric(df["SPPISO-East (BIDCLOSE_LOAD_FORECAST)"], errors="coerce")

    # net load actual
    df["net_load_actual"] = pd.to_numeric( df["SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE)"], errors="coerce")

    # wind actual
    wind_cols = [c for c in df.columns if "WINDGEN_HOURLY" in c]
    wind_numeric = (df[wind_cols].apply(pd.to_numeric, errors="coerce"))

    wind_sum = wind_numeric.sum(axis=1, min_count=1)

    all_zero = (wind_numeric.fillna(0).sum(axis=1).eq(0))

    df["wind_actual"] = wind_sum.mask(all_zero)

    # outage actual
    df["outage_ng"] = pd.to_numeric(df["SPPISO-East (NG_CAPACITY_OFFLINE)"], errors="coerce")
    df["outage_coal"] = pd.to_numeric( df["SPPISO-East (COAL_CAPACITY_OFFLINE)"], errors="coerce")

    # temperature actual
    temp_cols = [c for c in df.columns if "WSI_TRADER_FEELS_TEMP" in c]
    temp_numeric = (df[temp_cols].apply(pd.to_numeric, errors="coerce"))

    temp_avg = temp_numeric.mean(axis=1)

    all_zero_temp = (temp_numeric.fillna(0).sum(axis=1).eq(0))

    df["temperature_actual"] = temp_avg.mask(all_zero_temp)

    # windspeed actual
    wind_speed_cols = [c for c in df.columns if "WSI_TRADER_WIND" in c]
    df["wind_speed_actual"] = df[wind_speed_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    
    
    # outage actual
    df["total_outages"] = df["outage_ng"] + df["outage_coal"]

    out = df[["datetime", "load_actual", "net_load_actual", "wind_actual", "temperature_actual", "wind_speed_actual", "total_outages"]].dropna(subset=["datetime"])

    return out.sort_values("datetime").reset_index(drop=True)


In [12]:
start = "2026-07-01"
end = "2026-12-31"

df_actual = pull_yes_actual_historical(yes_username, yes_password, start_date=start, end_date = end)

Pulling actuals: 2026-07-01 ---> 2026-07-31
Pulling actuals: 2026-08-01 ---> 2026-08-31
Pulling actuals: 2026-09-01 ---> 2026-09-30
Pulling actuals: 2026-10-01 ---> 2026-10-31
Pulling actuals: 2026-11-01 ---> 2026-11-30
Pulling actuals: 2026-12-01 ---> 2026-12-31


In [13]:
df_actual_cleaned = clean_yes_actual(df_actual)

In [14]:
df_actual_cleaned[(df_actual_cleaned["datetime"]>="2026-08-05") & (df_actual_cleaned["datetime"]<"2026-08-06")]

,datetime,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages
840,2026-08-05 00:00:00,39410,20849.930000,17502.708,54.333333,2.85,4963.44
841,2026-08-05 01:00:00,37769,19816.660000,17786.953,52.666667,3.45,4963.44
842,2026-08-05 02:00:00,36564,19288.320000,17667.322,53.666667,4.05,4963.44
843,2026-08-05 03:00:00,35751,19484.370000,17225.528,52.333333,3.45,4963.44
844,2026-08-05 04:00:00,35380,20373.810000,14184.355,51.666667,4.05,4963.44
845,2026-08-05 05:00:00,35666,21747.360000,10774.332,50.333333,3.45,5092.44
846,2026-08-05 06:00:00,36532,23796.488333,8590.376,48.666667,4.05,5622.44
847,2026-08-05 07:00:00,37568,26065.889167,7450.333,54.000000,6.35,5622.44
848,2026-08-05 08:00:00,39022,28573.552500,6128.223,60.000000,6.90,5622.44
849,2026-08-05 09:00:00,40704,30794.871667,5033.610,66.000000,9.20,5477.54


In [16]:
df_actual_cleaned.to_csv('./output-data/yes actual cleaned.csv', index=False)
df_forecast_cleaned.to_csv('./output-data/yes forecast cleaned.csv', index=False)

In [18]:
df_forecast_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime                1200 non-null   datetime64[us]
 1   load_forecast           1020 non-null   float64       
 2   net_load_forecast       857 non-null    float64       
 3   wind_forecast           1200 non-null   float64       
 4   temperature_forecast    1200 non-null   float64       
 5   wind_speed_forecast     1200 non-null   float64       
 6   total_offline_forecast  1017 non-null   float64       
 7   offline_ng_forecast     1017 non-null   float64       
 8   offline_coal_forecast   1017 non-null   float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 84.5 KB
